# Simple RVC - Voice Conversion Demo

This notebook demonstrates how to use [Simple RVC](https://github.com/SawitProject/rvc) for voice conversion in Google Colab.

## Quick Steps
1. **Install** RVC
2. **Download** model from HuggingFace URL
3. **Upload** input audio
4. **Run** voice conversion
5. **Download** the result

In [ ]:
# @title **1. Install RVC** { display-mode: "form" }
# @markdown Installs RVC and dependencies. Takes about 2-3 minutes.
import subprocess

print("Installing FFmpeg...")
subprocess.run(["apt-get", "install", "-y", "ffmpeg"], capture_output=True)
print("FFmpeg installed.")

print("Installing RVC...")
!pip install git+https://github.com/SawitProject/rvc.git -q

try:
    from rvc.infer import __version__
    print(f"\nRVC installed successfully! Version: {__version__}")
except ImportError:
    print("\nRVC installed successfully!")

import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected. Using CPU (will be slower).")

In [ ]:
# @title **2. Download Model** { display-mode: "form" }
# @markdown Enter model name and URL. Supports .zip (auto-extract) or direct .pth files.
import os
import glob
import zipfile

# @markdown ### Model Info
model_name = "Applejack"  # @param {type:"string"}
model_url = "https://huggingface.co/marup/ApplejackRVC/resolve/main/Applejack.zip"  # @param {type:"string"}

models_dir = "/content/rvc_models"
model_folder = os.path.join(models_dir, model_name)
os.makedirs(model_folder, exist_ok=True)

# Download
ext = model_url.split("?")[0].split(".")[-1].lower()
download_path = os.path.join(model_folder, f"download.{ext}")

print(f"Downloading {model_name}...")
!wget -q "{model_url}" -O "{download_path}"

# Auto-extract if zip
if ext == "zip":
    print(f"Extracting zip to {model_folder}...")
    with zipfile.ZipFile(download_path, 'r') as z:
        z.extractall(model_folder)
    os.remove(download_path)
    print("Zip extracted and removed.")

# Auto-detect .pth and .index
pth_files = glob.glob(os.path.join(model_folder, "**", "*.pth"), recursive=True)
index_files = glob.glob(os.path.join(model_folder, "**", "*.index"), recursive=True)

model_path = pth_files[0] if pth_files else ""
index_path = index_files[0] if index_files else ""

print(f"\nModel folder: {model_folder}")
print(f"  .pth found:  {model_path}")
print(f"  .index found: {index_path or 'None'}")

if not model_path:
    print("\nWARNING: No .pth file found! Check the URL and model name.")

In [ ]:
# @title **3. Upload Input Audio** { display-mode: "form" }
# @markdown Upload an audio file (WAV, MP3, FLAC, etc.) or provide a URL.
from google.colab import files

audio_source = "upload"  # @param ["upload", "url"]

input_dir = "/content/rvc_input"
os.makedirs(input_dir, exist_ok=True)

if audio_source == "upload":
    print("Please upload your audio file:\n")
    uploaded = files.upload()
    for filename in uploaded.keys():
        src = filename
        dst = os.path.join(input_dir, filename)
        if src != dst:
            os.rename(src, dst)
        input_audio = dst
        print(f"  Saved: {dst}")
else:
    audio_url = ""  # @param {type:"string"}
    input_audio = os.path.join(input_dir, "input_audio")
    print(f"Downloading audio...")
    !wget -q "{audio_url}" -O "{input_audio}"
    print(f"  Saved: {input_audio}")

print(f"\nInput audio: {input_audio}")

In [ ]:
# @title **4. Run Voice Conversion** { display-mode: "form" }
# @markdown Configure settings and run the pipeline.
from rvc.infer.infer import run_inference_script
from rvc.lib.config import Config

# @markdown ### Voice Conversion Settings
pitch = 0  # @param {type:"slider", min:-24, max:24, step:1}
f0_method = "rmvpe"  # @param ["pm", "dio", "harvest", "yin", "pyin", "swipe", "rmvpe", "rmvpe-legacy", "fcpe", "fcpe-legacy", "crepe-tiny", "crepe-small", "crepe-medium", "crepe-large", "crepe-full", "mangio-crepe-tiny", "mangio-crepe-small", "mangio-crepe-medium", "mangio-crepe-large", "mangio-crepe-full", "djcm"]
index_rate = 0.5  # @param {type:"slider", min:0.0, max:1.0, step:0.05}
volume_envelope = 1.0  # @param {type:"slider", min:0.0, max:1.0, step:0.05}
protect = 0.5  # @param {type:"slider", min:0.0, max:0.5, step:0.05}
filter_radius = 3  # @param {type:"slider", min:1, max:10, step:1}
hop_length = 64  # @param {type:"slider", min:32, max:256, step:32}

# @markdown ### Advanced Features
enable_autotune = False  # @param {type:"boolean"}
autotune_strength = 1.0  # @param {type:"slider", min:0.0, max:1.0, step:0.05}
enable_clean_audio = False  # @param {type:"boolean"}
clean_strength = 0.7  # @param {type:"slider", min:0.0, max:1.0, step:0.05}
enable_formant_shifting = False  # @param {type:"boolean"}
formant_qfrency = 0.8  # @param {type:"slider", min:0.0, max:1.6, step:0.1}
formant_timbre = 0.8  # @param {type:"slider", min:0.0, max:1.6, step:0.1}

# @markdown ### Output Settings
output_format = "wav"  # @param ["wav", "flac", "mp3", "ogg"]
resample_sr = 0  # @param {type:"slider", min:0, max:48000, step:1000}

output_dir = "/content/rvc_output"
os.makedirs(output_dir, exist_ok=True)
output_audio = os.path.join(output_dir, f"output.{output_format}")

# Validate
if not model_path or not os.path.exists(model_path):
    raise FileNotFoundError(f"Model .pth not found: {model_path}. Run Step 2 first!")
if not os.path.exists(input_audio):
    raise FileNotFoundError(f"Input audio not found: {input_audio}. Run Step 3 first!")

print("=" * 50)
print("RVC Voice Conversion")
print("=" * 50)
print(f"  Model:      {model_path}")
print(f"  Index:      {index_path or 'None'}")
print(f"  Input:      {input_audio}")
print(f"  Output:     {output_audio}")
print(f"  Pitch:      {pitch:+d} semitones")
print(f"  F0 Method:  {f0_method}")
print(f"  Index Rate: {index_rate}")
print(f"  Autotune:   {'On (strength=' + str(autotune_strength) + ')' if enable_autotune else 'Off'}")
print(f"  Clean:      {'On (strength=' + str(clean_strength) + ')' if enable_clean_audio else 'Off'}")
print("=" * 50)

config = Config()

print("\nStarting voice conversion...\n")
run_inference_script(
    config=config,
    input_path=input_audio,
    output_path=output_audio,
    pth_path=model_path,
    index_path=index_path if index_path else None,
    pitch=pitch,
    f0_method=f0_method,
    index_rate=index_rate,
    volume_envelope=volume_envelope,
    protect=protect,
    filter_radius=filter_radius,
    hop_length=hop_length,
    export_format=output_format,
    embedder_model="contentvec_base",
    resample_sr=resample_sr,
    f0_autotune=enable_autotune,
    f0_autotune_strength=autotune_strength,
    split_audio=False,
    clean_audio=enable_clean_audio,
    clean_strength=clean_strength,
    formant_shifting=enable_formant_shifting,
    formant_qfrency=formant_qfrency,
    formant_timbre=formant_timbre,
)

if os.path.exists(output_audio):
    file_size = os.path.getsize(output_audio) / 1024 / 1024
    print(f"\nConversion complete! Output: {output_audio} ({file_size:.2f} MB)")
else:
    print("\nConversion may have failed. Check the output above for errors.")

In [ ]:
# @title **5. Play & Download Result** { display-mode: "form" }
# @markdown Listen to the converted audio and download it.
from IPython.display import Audio, display
from google.colab import files

if os.path.exists(output_audio):
    print("Playing converted audio:")
    display(Audio(output_audio))
    print("\nDownloading...")
    files.download(output_audio)
else:
    print("Output file not found. Did the conversion succeed?")

---

## Tips & Tricks

- **Model URL**: Use HuggingFace direct links like `https://huggingface.co/{user}/{model}/resolve/main/{file}.zip`
- **Model Name**: This becomes the folder name where the zip is extracted. Use any name you like.
- **Pitch**: Positive = higher voice, negative = lower. Start with small values (+/- 5).
- **F0 Method**: `rmvpe` = best quality. `pm` = fastest. `crepe-large` = highest accuracy.
- **Index Files**: Auto-detected from the model folder. Use `index_rate=0.5-0.75` for better similarity.
- **Autotune**: Great for singing. Set strength to 0.5-0.8 for natural results.
- **Hybrid F0**: Combine methods like `hybrid[rmvpe+fcpe]` for more robust pitch detection.

For more details, see the [documentation](https://github.com/SawitProject/rvc/blob/main/DOCUMENTATION.md).